## 2. Falsify a plausible reading

The provided bad source admits a binary form but supplies one argument
to an instance. Predict the source line and refusal. Then make a second
mutant: remove the instance from the good source. The `check derives`
assertion should fail because the required positive instance is absent.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from hashlib import sha256
from pathlib import Path


def find_repositories():
    candidates = [Path.cwd(), *Path.cwd().parents]
    extra = os.environ.get("GENESIS_REPOS_HOME")
    if extra:
        candidates.insert(0, Path(extra))
    for root in candidates:
        has_genesis = (root / "genesis" / "3_alpha_mathematical_language" / "Cargo.toml").is_file() or (root / "3_alpha_mathematical_language" / "Cargo.toml").is_file()
        has_symbol = (root / "the_symbol_game" / "omega_engine_education").is_dir() or (root / "omega_engine_education").is_dir()
        if has_genesis and has_symbol:
            g_root = root / "genesis" if (root / "genesis").is_dir() else root
            s_root = root / "the_symbol_game" if (root / "the_symbol_game").is_dir() else root
            return g_root, s_root
    raise RuntimeError(
        "Place genesis and the_symbol_game beside each other or set GENESIS_REPOS_HOME"
    )


GENESIS, SYMBOL = find_repositories()
REPOS_HOME = GENESIS.parent
ALPHA = Path(os.environ.get("GENESIS_ALPHA_ROOT", GENESIS / "3_alpha_mathematical_language"))
_PFP_PAPERS = (
    SYMBOL
    / "omega_engine_atlas"
    / "coda_atlas"
    / "F004_papers"
    / "015_pfp"
)
PAPER33 = _PFP_PAPERS / "005_pfp_practitioners_field_manual" / "code"
PAPER32 = _PFP_PAPERS / "004_pfp_native_verification_and_evidence_custody" / "code"


def digest(path):
    return sha256(Path(path).read_bytes()).hexdigest()


source_id = {
    "alpha_front_end_sha256": digest(ALPHA / "src" / "lib.rs"),
    "paper032_governor_sha256": digest(PAPER32 / "l1_symbol_game_governor.py"),
    "paper033_toolkit_sha256": digest(PAPER33 / "symbol_game_toolkit.py"),
}
print(json.dumps(source_id, indent=2))


In [ ]:
cargo = shutil.which("cargo")
if cargo is None:
    raise RuntimeError("Rust/Cargo is required for this source-backed lab")
TARGET = Path(tempfile.gettempdir()) / "genesis_alpha_omega_practicum_target"
build_env = dict(os.environ, CARGO_TARGET_DIR=str(TARGET))
build = subprocess.run(
    [cargo, "build", "--offline", "--quiet", "--bin", "alpha"],
    cwd=ALPHA,
    env=build_env,
    capture_output=True,
    text=True,
    timeout=180,
)
if build.returncode:
    raise RuntimeError("Alpha build refused: " + build.stderr)
binary = TARGET / "debug" / ("alpha.exe" if os.name == "nt" else "alpha")


def run_alpha(source):
    with tempfile.TemporaryDirectory(prefix="alpha-practicum-") as tmp:
        path = Path(tmp) / "lesson.alpha"
        path.write_text(source, encoding="utf-8")
        result = subprocess.run(
            [str(binary), str(path)], cwd=ALPHA, capture_output=True, text=True, timeout=30
        )
    return {
        "source_sha256": sha256(source.encode("utf-8")).hexdigest(),
        "binary_sha256": digest(binary),
        "exit": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    }


first_source = (ALPHA / "examples" / "first.alpha").read_text(encoding="utf-8")
first = run_alpha(first_source)
assert first["exit"] == 0, first["stderr"]
assert "replay=checked" in first["stdout"]
print(first["stdout"])

In [ ]:
bad_source = (ALPHA / "examples" / "refused.alpha").read_text(encoding="utf-8")
refused = run_alpha(bad_source)
assert refused["exit"] == 1 and "line 4" in refused["stderr"]
assert "form expects 2 arguments, got 1" in refused["stderr"]
assert "finished:" not in refused["stdout"]
print("first refusal:", refused["stderr"].strip())

mutant = first_source.replace("admit instance seen a\n", "")
assert mutant != first_source
no_witness = run_alpha(mutant)
assert no_witness["exit"] == 1 and "positive instance is not derivable" in no_witness["stderr"]
print("missing-witness refusal:", no_witness["stderr"].strip())

**Assessment B (3 points).** Change `learner_refusal` if your prediction
differed. Identify which failure is lexical/arity and which is an
unsatisfied host assertion. The CLI suppresses a success summary on
either failed run.


In [ ]:
learner_refusal = (
    "line 4",
    "form expects 2 arguments, got 1",
    "positive instance is not derivable",
)
score_b = sum(
    (
        learner_refusal[0] in refused["stderr"],
        learner_refusal[1] in refused["stderr"],
        learner_refusal[2] in no_witness["stderr"],
    )
)
assert score_b == 3
print("B:", score_b, "/3")

> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `13_alpha_to_omega_execution`
- The code checks one finite example. It is not a general proof.

## Exercise

Change one declared input in the code cell. The check should fail for the wrong object and pass for the finite example written here.
